<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH09/CH09_NB01_MoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# Rearchitecting LLMs
## Structural techniques for efficient models

### Chapter 9: Mixture of Experts (MoE)

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
Colab Environment: GPU T4

Models:
- HuggingFaceTB/SmolLM2-1.7B-Instruct

_____
This notebook builds a two-expert MoE adaptation on top of SmolLM2 to preserve general behavior while improving clinical extraction.
It trains a router and one domain expert, then compares soft vs hard routing through schema-compliance evaluation and token-level routing analysis.

# Setting up the notebook

This section installs dependencies, imports required modules, and defines utility helpers used across the notebook.

In [1]:
!pip install -q \
    transformers==5.0.0 \
    datasets==4.0.0 \
    accelerate==1.12.0 \

ERROR: Invalid requirement: '\\': Expected package name at the start of dependency specifier
    \
    ^


In [2]:
import copy
import json
import random
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
random.seed(0)
print("Running on:", DEVICE)

def clear_gpu_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

Running on: cuda


In [3]:
def set_seed(seed=42):
    """Set random seed for reproducibility"""
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✓ Random seed set to 42")

✓ Random seed set to 42


## Configuration

Centralized parameters for reproducible experiments. Adjust values here to rerun the notebook with different settings.

In [4]:
# Core model and training setup
MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
EPOCHS = 3
LAYERS_TO_MODIFY = range(13, 24)  # Last 11 layers of the 24-layer model

# Dataset sizes
CLINICAL_TRAIN_SAMPLES = 300
GENERAL_TRAIN_SAMPLES = 300

# Sequence and generation settings
MAX_TRAIN_LENGTH = 512
MAX_GENERAL_TEST_TOKENS = 50
MAX_CLINICAL_TEST_TOKENS = 150
MAX_EVAL_TOKENS = 400

# Reproducibility and runtime
SEED = 42
set_seed(SEED)
print(f"Configuration loaded | seed={SEED} | epochs={EPOCHS}")

Configuration loaded | seed=42 | epochs=3


## Base model and dataset

Load the base instruction model and the clinical training split used to build specialization prompts.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model.to(DEVICE)
print(f"Base model: {sum(p.numel() for p in model.parameters()):,} parameters")

# Clinical dataset used in Chapter 7
dataset = load_dataset("oopere/clinical-ner-qdora")
train_data = dataset["train"].select(range(CLINICAL_TRAIN_SAMPLES))
print(f"\nClinical dataset loaded: {len(train_data)} training examples")
print(f"Sample note: {train_data[0]['note'][:100]}...")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/102k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/360 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40 [00:00<?, ? examples/s]


Clinical dataset loaded: 300 training examples
Sample note: pt is 45 y.o male came in complainig of rly bad cogh and fevr for last 3 dys. says he feels super we...


## MoE block definition

Define a two-expert MoE module with a trainable router. During training it uses soft routing so gradients can update both router and expert parameters.

In [6]:
class Trainable2ExpertMoE(nn.Module):
    """Two-expert soft MoE block.

    Args:
        original_mlp: the MLP from the decoder layer. Becomes Expert 0 (general).
                      Expert 1 (clinical) is initialized as a deep copy of it.
        hidden_size: model hidden dimension, used to size the router.
    """

    def __init__(self, original_mlp, hidden_size):
        super().__init__()
        target_dtype = next(original_mlp.parameters()).dtype
        target_device = next(original_mlp.parameters()).device

        self.expert_general = original_mlp
        self.expert_clinical = copy.deepcopy(original_mlp)
        self.router = nn.Linear(hidden_size, 2, bias=False,
                                dtype=target_dtype, device=target_device)
        self.last_routing = None

    def forward(self, hidden_states):
        router_logits = self.router(hidden_states)

        if self.training:
            noise = torch.randn_like(router_logits) * 0.35
            router_logits = router_logits + noise

        # Upcast to FP32 for softmax numerical stability
        routing_weights = F.softmax(router_logits.float(), dim=-1).to(hidden_states.dtype)

        out_general = self.expert_general(hidden_states)
        out_clinical = self.expert_clinical(hidden_states)

        # Soft routing: each token receives a weighted blend of both experts
        output = out_general * routing_weights[..., 0].unsqueeze(-1) + out_clinical * routing_weights[..., 1].unsqueeze(-1)
        self.last_routing = routing_weights.detach()

        return output

## Layer surgery

Replace selected decoder MLP blocks with the MoE module so only deeper layers are specialized.

In [7]:
hidden_size = model.config.hidden_size

for layer_idx in LAYERS_TO_MODIFY:
    original_mlp = model.model.layers[layer_idx].mlp
    model.model.layers[layer_idx].mlp = Trainable2ExpertMoE(original_mlp, hidden_size)

print(f"Surgery complete: {len(LAYERS_TO_MODIFY)} MLP layers replaced with MoE blocks.")

Surgery complete: 11 MLP layers replaced with MoE blocks.


## Freezing strategy

Freeze the base model and train only the clinical expert and the router in each modified layer.

In [8]:
# Freeze the entire model first
for param in model.parameters():
    param.requires_grad_(False)

# Unfreeze Expert 1 (clinical) and the router in each MoE block
for layer_idx in LAYERS_TO_MODIFY:
    for param in model.model.layers[layer_idx].mlp.expert_clinical.parameters():
        param.requires_grad_(True)
    for param in model.model.layers[layer_idx].mlp.router.parameters():
        param.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters              : {total:,}")
print(f"Trainable (Expert 1 + router) : {trainable:,}  ({trainable/total:.2%})")

Total parameters              : 2,265,069,568
Trainable (Expert 1 + router) : 553,693,184  (24.44%)


## Training data assembly

Create a mixed prompt set: clinical extraction examples plus general SmolTalk prompts, then shuffle for joint training.

In [9]:
# Clinical prompts: exact format used in Chapter 7 fine-tuning
clinical_prompts = []
for item in train_data:
    label = json.loads(item["label"]) if isinstance(item["label"], str) else item["label"]
    messages = [
        {"role": "system", "content": "Extract:"},
        {"role": "user", "content": item["note"]},
        {"role": "assistant", "content": json.dumps(label, indent=2)},
    ]
    clinical_prompts.append(
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    )

# General prompts from SmolTalk
smoltalk = load_dataset("HuggingFaceTB/smoltalk", "all", split="train")
smoltalk_single = smoltalk.filter(
    lambda x: len(x["messages"]) == 2 and len(x["messages"][0]["content"]) < 300
)
general_subset = smoltalk_single.select(range(GENERAL_TRAIN_SAMPLES))

general_prompts = []
for item in general_subset:
    general_prompts.append(
        tokenizer.apply_chat_template(
            item["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    )

training_prompts = clinical_prompts + general_prompts
random.shuffle(training_prompts)
print(
    f"Training prompts - clinical: {len(clinical_prompts)} | "
    f"general: {len(general_prompts)} | total: {len(training_prompts)}"
)

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/all/train-00000-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00001-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00002-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00003-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

...

data/all/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/54948 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Training prompts - clinical: 300 | general: 300 | total: 600


## MoE training function

Define the training routine for router and clinical experts using mixed-precision optimization.

In [10]:
def train_MoE_router_expert(model):
    model.config.use_cache = False
    model.to(DEVICE).train()

    # Promote trainable parameters to FP32 for optimizer stability
    for param in model.parameters():
        if param.requires_grad:
            param.data = param.data.float()

    router_params = []
    expert_params = []

    for layer_idx in LAYERS_TO_MODIFY:
        router_params.extend(model.model.layers[layer_idx].mlp.router.parameters())
        expert_params.extend(model.model.layers[layer_idx].mlp.expert_clinical.parameters())

    optimizer = optim.AdamW([
        {"params": router_params, "lr": 1e-6},
        {"params": expert_params, "lr": 1e-5},
    ])

    scaler = GradScaler("cuda")

    for epoch in range(EPOCHS):
        total_loss = 0
        random.shuffle(training_prompts)

        for idx, text in enumerate(training_prompts):
            inputs = tokenizer(
                text, return_tensors="pt", truncation=True, max_length=MAX_TRAIN_LENGTH
            ).to(DEVICE)

            optimizer.zero_grad()

            with autocast("cuda", dtype=torch.float16):
                loss = model(**inputs, labels=inputs["input_ids"]).loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            if (idx + 1) % 100 == 0:
                print(
                    f"  Epoch {epoch+1}  step {idx+1}/{len(training_prompts)}  loss: {loss.item():.4f}"
                )

        print(f"-> END EPOCH {epoch+1:02d}  avg loss: {total_loss/len(training_prompts):.4f}\n")

    model.config.use_cache = True
    print("Router training complete.")

In [11]:
clear_gpu_cache()

## Training execution

Run optimization over the mixed prompt dataset to specialize the clinical expert and train routing behavior.

In [12]:
train_MoE_router_expert(model)

  Epoch 1  step 100/600  loss: 0.4501
  Epoch 1  step 200/600  loss: 0.5898
  Epoch 1  step 300/600  loss: 0.2235
  Epoch 1  step 400/600  loss: 0.5624
  Epoch 1  step 500/600  loss: 0.3824
  Epoch 1  step 600/600  loss: 0.3012
-> END EPOCH 01  avg loss: 0.6782

  Epoch 2  step 100/600  loss: 0.4068
  Epoch 2  step 200/600  loss: 0.2764
  Epoch 2  step 300/600  loss: 0.2882
  Epoch 2  step 400/600  loss: 0.7366
  Epoch 2  step 500/600  loss: 0.8577
  Epoch 2  step 600/600  loss: 0.2205
-> END EPOCH 02  avg loss: 0.4853

  Epoch 3  step 100/600  loss: 0.2183
  Epoch 3  step 200/600  loss: 0.8441
  Epoch 3  step 300/600  loss: 0.7235
  Epoch 3  step 400/600  loss: 0.4737
  Epoch 3  step 500/600  loss: 0.3887
  Epoch 3  step 600/600  loss: 0.4551
-> END EPOCH 03  avg loss: 0.3937

Router training complete.


## A/B generation and routing inspection

Evaluate qualitative behavior on a general prompt and a clinical prompt, then inspect per-token expert routing.

In [13]:
model.to(torch.float16)
model.eval()

def run_moe(prompt, max_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

# Test A - general knowledge
messages_general = [{"role": "user", "content": "What do you know about la Vall Fosca?"}]
prompt_general = tokenizer.apply_chat_template(
    messages_general, tokenize=False, add_generation_prompt=True
)
print("[Test A - General knowledge]")
print(f"Response: {run_moe(prompt_general, max_tokens=MAX_GENERAL_TEST_TOKENS)}\n")

# Test B - clinical NER
nota_test = (
    "Patient is a 50 yo F c/o severe chest pain and SOB since yesterday. "
    "Vitals: HR 110, BP 150/90. Started on Aspirin 81mg."
)
messages_clinical = [
    {"role": "system", "content": "Extract:"},
    {"role": "user", "content": nota_test},
]
prompt_clinical = tokenizer.apply_chat_template(
    messages_clinical, tokenize=False, add_generation_prompt=True
)
print("[Test B - Clinical NER]")
print(f"Response: {run_moe(prompt_clinical, max_tokens=MAX_CLINICAL_TEST_TOKENS)}")

[Test A - General knowledge]
Response: La Vall Fosca is a picturesque village located in the province of Gerona, Spain. It is situated in the Montseny Natural Park, which is a UNESCO Biosphere Reserve. The village is known for its stunning natural beauty, with

[Test B - Clinical NER]
Response: {
  "patient_age": 50,
  "symptoms": [
    "chest pain",
    "shortness of breath"
  ],
  "vital_signs": {
    "temperature": null,
    "heart_rate": 110,
    "blood_pressure": "150/90"
  },
  "medications": [
    {
      "name": "Aspirin",
      "dose": "81mg",
      "frequency": "not specified"
    }
  ],
  "duration_days": 1
}


In [14]:
def analyze_routing(text, layer_index=0):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _ = model(**inputs)

    moe_layer      = model.model.layers[layer_index].mlp
    routing_weights = moe_layer.last_routing.squeeze(0)
    tokens          = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    print(f"Routing analysis (layer {layer_index}):")
    print(f"{'Token':<18} | {'Expert 0 (General)':<20} | {'Expert 1 (Clinical)'}")
    print("-" * 65)
    for token, weights in zip(tokens, routing_weights):
        clean = token.replace("Ġ", "").replace(" ", "")
        print(f"{clean:<18} | {weights[0].item():<20.4f} | {weights[1].item():.4f}")

analyze_routing("Paris is the capital of France", layer_index=23)


Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Paris              | 0.4797               | 0.5200
is                 | 0.5571               | 0.4431
the                | 0.5996               | 0.4001
capital            | 0.5537               | 0.4463
of                 | 0.5166               | 0.4834
France             | 0.5801               | 0.4199


In [15]:
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=23)

Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Patient            | 0.5166               | 0.4834
is                 | 0.5044               | 0.4958
a                  | 0.5283               | 0.4717
                   | 0.5029               | 0.4971
6                  | 0.5112               | 0.4890
7                  | 0.5259               | 0.4741
-                  | 0.5225               | 0.4773
year               | 0.5996               | 0.4004
-                  | 0.5718               | 0.4282
old                | 0.4641               | 0.5356
male               | 0.4600               | 0.5400
with               | 0.3794               | 0.6206
chronic            | 0.4678               | 0.5322
pain               | 0.5479               | 0.4519


In [16]:
analyze_routing("Patient is a 67-year-old male with chronic back pain", layer_index=23)

Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Patient            | 0.5166               | 0.4834
is                 | 0.5044               | 0.4958
a                  | 0.5283               | 0.4717
                   | 0.5029               | 0.4971
6                  | 0.5112               | 0.4890
7                  | 0.5259               | 0.4741
-                  | 0.5225               | 0.4773
year               | 0.5996               | 0.4004
-                  | 0.5718               | 0.4282
old                | 0.4641               | 0.5356
male               | 0.4600               | 0.5400
with               | 0.3794               | 0.6206
chronic            | 0.4678               | 0.5322
back               | 0.5537               | 0.4465
pain               | 0.5532               | 0.4465


In [17]:
import pandas as pd

REQUIRED_SCHEMA = {
    "patient_age": (int, type(None)),
    "symptoms": list,
    "vital_signs": dict,
    "medications": list,
    "duration_days": (int, type(None)),
}
VITAL_SIGNS_KEYS = {"temperature", "heart_rate", "blood_pressure"}

def check_schema_compliance(response_text: str) -> dict:
    cleaned = response_text.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return {
            "is_valid_json": False,
            "matches_schema": False,
            "failure_reason": f"JSON parse error: {e}",
        }

    for key, expected_type in REQUIRED_SCHEMA.items():
        if key not in parsed:
            return {
                "is_valid_json": True,
                "matches_schema": False,
                "failure_reason": f"Missing key: '{key}'",
            }
        if not isinstance(parsed[key], expected_type):
            return {
                "is_valid_json": True,
                "matches_schema": False,
                "failure_reason": f"Wrong type for '{key}'",
            }

    for vk in VITAL_SIGNS_KEYS:
        if vk not in parsed.get("vital_signs", {}):
            return {
                "is_valid_json": True,
                "matches_schema": False,
                "failure_reason": f"Missing vital_signs key: '{vk}'",
            }

    return {"is_valid_json": True, "matches_schema": True, "failure_reason": None}

def evaluate_moe(label="MoE_Soft"):
    model.eval()
    results = []
    test_dataset = dataset["test"]

    for sample in test_dataset:
        messages = [
            {"role": "system", "content": "Extract:"},
            {"role": "user", "content": sample["note"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        response = run_moe(prompt, max_tokens=MAX_EVAL_TOKENS)
        compliance = check_schema_compliance(response)

        results.append(
            {
                "category": sample["category"],
                "model": label,
                "is_valid_json": compliance["is_valid_json"],
                "matches_schema": compliance["matches_schema"],
                "failure_reason": compliance["failure_reason"],
            }
        )

    return pd.DataFrame(results)

def summarize(df):
    summary = df.groupby("category").agg(
        samples=("is_valid_json", "count"),
        valid_json=("is_valid_json", "sum"),
        schema_ok=("matches_schema", "sum"),
    )
    summary["valid_json_pct"] = (summary["valid_json"] / summary["samples"] * 100).round(1)
    summary["schema_ok_pct"] = (summary["schema_ok"] / summary["samples"] * 100).round(1)
    return summary

print("Evaluating - soft inference...")
soft_results = evaluate_moe(label="MoE_Soft")
print(summarize(soft_results)[["samples", "valid_json_pct", "schema_ok_pct"]].to_string())
print(f"\nOverall schema compliance (soft): {soft_results['matches_schema'].mean()*100:.1f}%")

Evaluating - soft inference...
               samples  valid_json_pct  schema_ok_pct
category                                             
abbreviations        6           100.0          100.0
clean                8           100.0          100.0
implicit             7           100.0          100.0
irrelevant          12           100.0           91.7
typos                7           100.0          100.0

Overall schema compliance (soft): 97.5%


## Hard routing adaptation (inference only)

Swap the MoE forward behavior to hard token-to-expert decisions at inference time, preserving already-trained weights.

In [18]:
# The weights stay exactly the same. Only inference behavior changes.
class Trainable2ExpertMoE(nn.Module):
    def __init__(self, original_mlp, hidden_size):
        super().__init__()
        target_dtype = next(original_mlp.parameters()).dtype
        target_device = next(original_mlp.parameters()).device
        self.expert_general = original_mlp
        self.expert_clinical = copy.deepcopy(original_mlp)
        self.router = nn.Linear(hidden_size, 2, bias=False,
                                dtype=target_dtype, device=target_device)
        self.last_routing = None

    def forward(self, hidden_states):
        router_logits = self.router(hidden_states)
        routing_weights = F.softmax(router_logits.float(), dim=-1).to(hidden_states.dtype)

        if self.training:
            # Soft routing - gradients flow through the weighted sum
            out_general = self.expert_general(hidden_states)
            out_clinical = self.expert_clinical(hidden_states)
            output = out_general * routing_weights[..., 0].unsqueeze(-1) + out_clinical * routing_weights[..., 1].unsqueeze(-1)
            self.last_routing = routing_weights.detach()
        else:
            # Hard routing - each token goes to exactly one expert
            expert_choice = torch.argmax(router_logits, dim=-1)
            mask_clinical = (expert_choice == 1).unsqueeze(-1)

            # Both experts are still computed for all tokens for pedagogical clarity
            out_general = self.expert_general(hidden_states)
            out_clinical = self.expert_clinical(hidden_states)
            output = torch.where(mask_clinical, out_clinical, out_general)
            hard_decisions = F.one_hot(expert_choice, num_classes=2).float()
            self.last_routing = hard_decisions.detach()

        return output

# Patch existing MoE layers with the new forward method while preserving weights
for layer_idx in LAYERS_TO_MODIFY:
    old_block = model.model.layers[layer_idx].mlp
    new_block = Trainable2ExpertMoE(old_block.expert_general, hidden_size)
    new_block.expert_clinical.load_state_dict(old_block.expert_clinical.state_dict())
    new_block.router.load_state_dict(old_block.router.state_dict())
    model.model.layers[layer_idx].mlp = new_block

print("Forward method updated. Weights unchanged.")
model.eval()

Forward method updated. Weights unchanged.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-12): 13 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
      (13-23): 11 x LlamaDecoderLayer(
   

In [19]:
analyze_routing("Paris is the capital of France", layer_index=23)


Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Paris              | 1.0000               | 0.0000
is                 | 1.0000               | 0.0000
the                | 1.0000               | 0.0000
capital            | 1.0000               | 0.0000
of                 | 1.0000               | 0.0000
France             | 1.0000               | 0.0000


In [20]:
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=23)

Routing analysis (layer 23):
Token              | Expert 0 (General)   | Expert 1 (Clinical)
-----------------------------------------------------------------
Patient            | 1.0000               | 0.0000
is                 | 0.0000               | 1.0000
a                  | 1.0000               | 0.0000
                   | 0.0000               | 1.0000
6                  | 0.0000               | 1.0000
7                  | 1.0000               | 0.0000
-                  | 1.0000               | 0.0000
year               | 1.0000               | 0.0000
-                  | 1.0000               | 0.0000
old                | 0.0000               | 1.0000
male               | 0.0000               | 1.0000
with               | 0.0000               | 1.0000
chronic            | 0.0000               | 1.0000
pain               | 1.0000               | 0.0000


In [21]:
model.to(torch.float16)
model.eval()  # self.training = False -> hard routing

print("Evaluating - hard inference...")
hard_results = evaluate_moe(label="MoE_Hard")
print(summarize(hard_results)[["samples", "valid_json_pct", "schema_ok_pct"]].to_string())
print(f"\nOverall schema compliance (hard): {hard_results['matches_schema'].mean()*100:.1f}%")

# Side-by-side comparison
print("\n--- Soft vs Hard routing ---")
print(f"Soft: {soft_results['matches_schema'].mean()*100:.1f}%")
print(f"Hard: {hard_results['matches_schema'].mean()*100:.1f}%")

Evaluating - hard inference...
               samples  valid_json_pct  schema_ok_pct
category                                             
abbreviations        6           100.0          100.0
clean                8           100.0          100.0
implicit             7           100.0          100.0
irrelevant          12           100.0           91.7
typos                7           100.0          100.0

Overall schema compliance (hard): 97.5%

--- Soft vs Hard routing ---
Soft: 97.5%
Hard: 97.5%


# Summary and conclusions

This notebook demonstrates how a lightweight two-expert MoE adaptation can preserve general behavior while specializing for clinical extraction.
The soft and hard routing evaluations provide a practical comparison of quality, schema compliance, and routing interpretability.